# Airline Passenger Satisfaction

**Project:** Airline Satisfaction Insight Analyzer  
**Dataset:** Airline Passenger Satisfaction

---

## Contents
1. Setup & Imports
2. Data Loading & Preview
3. Data Cleaning
4. Univariate Analysis
5. Bivariate Analysis (by Satisfaction)
6. Correlation Heatmap
7. Service Ratings Deep Dive
8. Machine Learning Models
9. Feature Importance
10. Summary


---
## 1. Setup & Imports

In [ ]:
# Standard library
import sys
import warnings
warnings.filterwarnings('ignore')

# Ensure the project root is on the path
sys.path.insert(0, '..')

# Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)

# Project modules
from src import config
from src.data_processing import (
    load_dataset, standardize_column_names, clean_data,
    detect_feature_types, split_data, get_dataset_statistics
)
from src.visualization import (
    plot_satisfaction_distribution,
    plot_satisfaction_by_customer_type,
    plot_satisfaction_by_travel_type,
    plot_satisfaction_by_class,
    plot_flight_distance_distribution,
    plot_correlation_heatmap,
    plot_service_ratings,
    plot_age_distribution,
)
from src.model_training import (
    build_random_forest_pipeline,
    build_logistic_regression_pipeline,
    train_model, get_feature_importance
)
from src.evaluation import compute_metrics, plot_confusion_matrix, compare_models

print('Setup complete!')

---
## 2. Data Loading & Preview

In [ ]:
# Load raw dataset
df_raw = load_dataset()
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Descriptive statistics for numeric columns
df_raw.describe().T.style.background_gradient(cmap='Blues', axis=1)

In [ ]:
# Missing values summary
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

---
## 3. Data Cleaning

In [ ]:
# Standardise column names → snake_case
df = standardize_column_names(df_raw)
print('Columns after standardisation:')
print(list(df.columns))

In [ ]:
# Full cleaning pipeline
df = clean_data(df)
print(f'Clean dataset shape: {df.shape}')
df.head()

In [ ]:
# Target distribution
vc = df[config.TARGET_COLUMN].value_counts()
labels = {1: 'Satisfied', 0: 'Neutral / Dissatisfied'}
vc.index = vc.index.map(labels)
print('Target class distribution:')
print(vc)
print(f'\nSatisfaction rate: {df[config.TARGET_COLUMN].mean()*100:.1f}%')

---
## 4. Univariate Analysis

In [ ]:
# Overall satisfaction distribution
fig = plot_satisfaction_distribution(df)
plt.show()

In [ ]:
# Age distribution by satisfaction
fig = plot_age_distribution(df)
if fig:
    plt.show()

In [ ]:
# Flight distance distribution by satisfaction
fig = plot_flight_distance_distribution(df)
if fig:
    plt.show()

---
## 5. Bivariate Analysis (by Satisfaction)

In [ ]:
# Satisfaction by customer type
fig = plot_satisfaction_by_customer_type(df)
if fig:
    plt.show()

In [ ]:
# Satisfaction by type of travel
fig = plot_satisfaction_by_travel_type(df)
if fig:
    plt.show()

In [ ]:
# Satisfaction by travel class
fig = plot_satisfaction_by_class(df)
if fig:
    plt.show()

**Observations:**
- Loyal customers tend to have higher satisfaction rates.
- Business travel passengers show notably higher satisfaction than personal travel.
- Business class passengers are significantly more satisfied than Economy class.

---
## 6. Correlation Heatmap

In [ ]:
fig = plot_correlation_heatmap(df)
plt.show()

**Observations:**
- Several service rating variables show moderate positive correlation with overall satisfaction.
- Arrival and departure delay are positively correlated with each other (expected).
- Service ratings tend to cluster together, suggesting passengers rate holistically.

---
## 7. Service Ratings Deep Dive

In [ ]:
fig = plot_service_ratings(df)
if fig:
    plt.show()

In [ ]:
# Detailed statistics for service columns
rating_keywords = [
    'inflight', 'food', 'service', 'comfort', 'cleanliness',
    'entertainment', 'wifi', 'boarding', 'baggage', 'checkin',
    'legroom', 'seat', 'gate', 'departure', 'arrival'
]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
rating_cols = [c for c in num_cols if any(kw in c for kw in rating_keywords)
               and c != config.TARGET_COLUMN]

if rating_cols:
    target = config.TARGET_COLUMN
    rating_means = df.groupby(target)[rating_cols].mean()
    rating_means.index = rating_means.index.map({1: 'Satisfied', 0: 'Neutral/Dissatisfied'})
    rating_means.T.style.background_gradient(cmap='RdYlGn', axis=1)

---
## 8. Machine Learning Models

We train two classification models:
- **Random Forest** — non-linear ensemble, robust to outliers
- **Logistic Regression** — linear baseline, highly interpretable

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = split_data(df)
print(f'Training samples : {len(X_train):,}')
print(f'Testing  samples : {len(X_test):,}')
print(f'\nClass balance (train):\n{y_train.value_counts(normalize=True).round(3)}')

In [ ]:
%%time
# Train Random Forest
rf_pipeline = build_random_forest_pipeline(X_train)
rf_pipeline = train_model(rf_pipeline, X_train, y_train, model_name='RandomForest')
print('Training complete!')

In [ ]:
%%time
# Train Logistic Regression
lr_pipeline = build_logistic_regression_pipeline(X_train)
lr_pipeline = train_model(lr_pipeline, X_train, y_train, model_name='LogisticRegression')
print('Training complete!')

In [ ]:
# Evaluate both models
rf_pred = rf_pipeline.predict(X_test)
lr_pred = lr_pipeline.predict(X_test)

try:
    rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]
except Exception:
    rf_prob = None
try:
    lr_prob = lr_pipeline.predict_proba(X_test)[:, 1]
except Exception:
    lr_prob = None

rf_metrics = compute_metrics(y_test, rf_pred, rf_prob, 'Random Forest')
lr_metrics = compute_metrics(y_test, lr_pred, lr_prob, 'Logistic Regression')

In [ ]:
# Side-by-side comparison
comparison_df = pd.DataFrame([
    {k: v for k, v in rf_metrics.items() if k not in ('model', 'classification_report')},
    {k: v for k, v in lr_metrics.items() if k not in ('model', 'classification_report')},
], index=['Random Forest', 'Logistic Regression'])

comparison_df.style.highlight_max(axis=0, color='#c8f7c5').format(precision=4)

In [ ]:
# Confusion matrices
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Neutral/\nDissatisfied', 'Satisfied']

for ax, pred, title in [
    (axes[0], rf_pred, 'Random Forest'),
    (axes[1], lr_pred, 'Logistic Regression'),
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(8, 6))
for prob, label, color in [
    (rf_prob, 'Random Forest', '#2ECC71'),
    (lr_prob, 'Logistic Regression', '#3498DB'),
]:
    if prob is not None:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1], [0,1], 'k--', lw=1)
ax.set_title('ROC Curves — Model Comparison', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Feature Importance

In [ ]:
# Extract Random Forest feature importances
importance_df = get_feature_importance(rf_pipeline, X_train, top_n=20)
importance_df.head(10)

In [ ]:
# Plot top 20 features
if not importance_df.empty:
    plot_df = importance_df.copy()
    plot_df['feature'] = (
        plot_df['feature']
        .str.replace(r'^num__|^cat__', '', regex=True)
        .str.replace('_', ' ')
        .str.title()
    )
    plot_df = plot_df.sort_values('importance')

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(plot_df['feature'], plot_df['importance'],
                   color='#2ECC71', edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', fontsize=8, padding=3)
    ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

---
## 10. Summary

### Key Findings

| Dimension | Finding |
|-----------|--------|
| Overall satisfaction | ~54% satisfied across dataset |
| Customer type | Loyal customers more satisfied |
| Travel type | Business travellers more satisfied |
| Travel class | Business class highest satisfaction |
| Top feature | In-flight Wi-Fi / entertainment / seat comfort |
| Best ML model | Random Forest (higher F1 & AUC) |

### Recommendations for Airlines
1. Prioritise in-flight Wi-Fi quality and entertainment content
2. Improve seat comfort for Economy class passengers
3. Reduce departure and arrival delays
4. Tailor service for personal-travel passengers who show lower satisfaction

### Next Steps
- Run `python main.py` to execute the full automated pipeline
- Check `outputs/reports/ai_insight_report.md` for the AI-generated executive summary
- Inspect `outputs/figures/` for all saved visualisations

